In [1]:
%pip install jupysql sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [64]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine

df_us = pd.read_csv('C:/Users/alex/Study/SQLЗапросы/users.csv')
df_web = pd.read_csv('C:/Users/alex/Study/SQLЗапросы/webinar.csv')
df_trans = pd.read_csv('C:/Users/alex/Study/SQLЗапросы/transactions.csv')
engine = create_engine('sqlite:///UsersWeb.db')

df_us.to_sql('users', con=engine, if_exists='replace', index=False)
df_web.to_sql('Email', con=engine, if_exists='replace', index=False)
df_trans.to_sql('transactions', con=engine, if_exists='replace', index=False)

12

In [65]:
engine

Engine(sqlite:///UsersWeb.db)

In [74]:
with engine.connect() as connection:
    query_result = pd.read_sql("SELECT * FROM users", con=connection)
    print(query_result)
with engine.connect() as connection:
    query_result = pd.read_sql("SELECT * FROM Email", con=connection)
    print(query_result)
with engine.connect() as connection:
    query_result = pd.read_sql("SELECT * FROM transactions", con=connection)
    print(query_result)

   user_id                   email    date_registration
0   456451             s_k76@bk.ru  2016-06-08 15:00:00
1   456481             desbo@ya.ru  2016-04-22 15:00:00
2    46545             s_k76@bk.ru  2016-04-21 15:00:00
3   456281          tte-84@mail.ru  2016-03-22 15:00:00
4    44545             s_k76@bk.ru  2016-05-02 15:00:00
5   123213      sekretshop@mail.ru  2016-06-02 15:00:00
6    35656  waitingforward@mail.ru  2016-03-02 15:00:00
7    35657  waitingforward@mail.ru  2016-05-02 15:00:00
                         email
0                    n3p@bk.ru
1                  s_k76@bk.ru
2            liverpool64@bk.ru
3          reklamagrad@mail.ru
4           serdepar@yandex.ru
5  academi@academy-trading.com
6               np1968@mail.ru
7            ixdir@dimaxweb.ru
8              9169399@mail.ru
9       waitingforward@mail.ru
    user_id  price
0     43243   1000
1    456281   1000
2    456451   1000
3     46545   1000
4     44545   1000
5     43343   1000
6     43343   1000
7  

In [73]:
target_date = "2016-04-01 00:00:00"
query = f"""
    WITH unique_email AS (
        SELECT 
            email,
            MIN(user_id) AS user_id,
            MIN(date_registration) AS first_reg
        FROM users
        WHERE email IN Email
        GROUP BY email
        HAVING MIN(date_registration) > '{target_date}'
    )
    SELECT 
        e.user_id, 
        e.email, 
        e.first_reg AS date_registration,
        SUM(t.price) AS total_spent
    FROM unique_email e
    LEFT JOIN transactions t ON e.user_id = t.user_id
    GROUP BY e.user_id, e.email, e.first_reg
"""

with engine.connect() as connection:
    new_users = pd.read_sql(query, con=connection)

print(new_users)


   user_id        email    date_registration  total_spent
0    44545  s_k76@bk.ru  2016-04-21 15:00:00         1000
